# Limpieza de Inconsistencias — NovaMarket

Autor: Daniel Pallarés

Dataset de entrada: S04_PD_Pallares_DatasetImputacionInicial.csv (640,000 filas, salida de la Misión 1).

Objetivo: dejar limpias canal_compra, metodo_pago, categoria_producto y ciudad_tienda, convertir fecha_pedido a fecha real, y resolver los duplicados que ya había encontrado en la auditoría. Aquí no vuelvo a diagnosticar, corrijo lo que ya diagnostiqué en S03_PD_Pallares_AuditoriaCalidad.ipynb.

In [1]:
import pandas as pd
import numpy as np
import unicodedata

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

df = pd.read_csv('S04_PD_Pallares_DatasetImputacionInicial.csv')
N = len(df)
print(f"Filas: {N:,} | Columnas: {df.shape[1]}")
df.head()

Filas: 640,000 | Columnas: 25


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente,precio_monto_imputado,nivel_satisfaccion_imputado,ciudad_tienda_imputado,correo_cliente_imputado,comentario_cliente_imputado
0,P505195,C45650,15/11/2025,Tienda,Efectivo,bogota,1100103,Moda,Chaqueta impermeable,28700.0,1,28700.0,32,cliente405195@novamarket.com.co,Bajo,Platino,El domiciliario fue muy amable,14.9,2026-04-17,0,False,False,False,False,False
1,P677836,C84260,2026-03-05,Web,Efectivo,MZL,1700101,Electrónica,Parlante bluetooth,195700.0,2,391400.0,73,cliente577836@yahoo.es,Medio,Bronce,El producto llegó dañado,12.9,2026-06-10,0,False,False,False,False,False
2,P387869,C164166,2025-03-13,Tienda,PayPal,Cúcuta,5400101,Juguetería,Rompecabezas 1000 piezas,597600.0,-3,-1792800.0,32,cliente287869@novamarket.com.co,Bajo,Oro,El domiciliario fue muy amable,14.3,2026-02-26,0,False,False,False,False,False
3,P481069,C98406,27/06/2025,App,transferencia,Cucuta,5400103,deportes,Colchoneta de yoga,322900.0,1,322900.0,45,cliente381069@gmail.com,Medio,Platino,Rápido y sin problemas,9.5,2026-05-14,0,False,False,False,False,False
4,P247920,C163018,01/05/2025,Web,Efectivo,Bucaramanga,6800112,Hogar,Juego de sábanas,222400.0,1,222400.0,60,cliente147920@novamarket.com.co,Alto,Plata,Sin comentario,12.3,2026-07-13,0,False,False,False,False,True


## Paso 1 — Valores únicos antes de corregir

Reviso los valores únicos de las 4 columnas de texto antes de tocar nada, para tener con qué comparar después.

In [2]:
columnas_texto = ['canal_compra', 'metodo_pago', 'categoria_producto', 'ciudad_tienda']

for col in columnas_texto:
    print(f"{col}: {df[col].nunique(dropna=True)} valores unicos crudos")

print()
print("canal_compra:", sorted(df['canal_compra'].dropna().unique()))
print()
print("metodo_pago:", sorted(df['metodo_pago'].dropna().unique()))
print()
print("categoria_producto:", sorted(df['categoria_producto'].dropna().unique()))
print()
print(f"ciudad_tienda ({df['ciudad_tienda'].nunique()} valores unicos):", sorted(df['ciudad_tienda'].dropna().unique()))

canal_compra: 20 valores unicos crudos
metodo_pago: 20 valores unicos crudos
categoria_producto: 32 valores unicos crudos
ciudad_tienda: 88 valores unicos crudos

canal_compra: [' App', ' Marketplace', ' Tienda', ' Web', 'APP', 'App', 'App ', 'MARKETPLACE', 'Marketplace', 'Marketplace ', 'TIENDA', 'Tienda', 'Tienda ', 'WEB', 'Web', 'Web ', 'app', 'marketplace', 'tienda', 'web']



metodo_pago: [' Efectivo', ' PayPal', ' Tarjeta', ' Transferencia', 'EFECTIVO', 'Efectivo', 'Efectivo ', 'PAYPAL', 'PayPal', 'PayPal ', 'TARJETA', 'TRANSFERENCIA', 'Tarjeta', 'Tarjeta ', 'Transferencia', 'Transferencia ', 'efectivo', 'paypal', 'tarjeta', 'transferencia']

categoria_producto: [' Belleza', ' Deportes', ' Electrónica', ' Hogar', ' Juguetería', ' Moda', 'BELLEZA', 'Belleza', 'Belleza ', 'DEPORTES', 'Deportes', 'Deportes ', 'ELECTRÓNICA', 'Electrónica', 'Electrónica ', 'HOGAR', 'Hogar', 'Hogar ', 'JUGUETERÍA', 'Juguetería', 'Juguetería ', 'MODA', 'Moda', 'Moda ', 'belleza', 'deportes', 'electronica', 'electrónica', 'hogar', 'jugueteria', 'juguetería', 'moda']



ciudad_tienda (88 valores unicos): [' Barranquilla', ' Bogotá', ' Bucaramanga', ' Cali', ' Cartagena', ' Cúcuta', ' Ibagué', ' Manizales', ' Medellín', ' Pereira', ' Santa Marta', ' Villavicencio', 'B/manga', 'B/quilla', 'BAQ', 'BARRANQUILLA', 'BGA', 'BOG', 'BOGOTA', 'BOGOTA DC', 'BOGOTÁ', 'BUCARAMANGA', 'Barranquilla', 'Barranquilla ', 'Bogota', 'Bogota D.C.', 'Bogotá', 'Bogotá ', 'Bucaramanga', 'Bucaramanga ', 'CALI', 'CALI ', 'CARTAGENA', 'CTG', 'CUCUTA', 'Cali', 'Cali ', 'Cartagena', 'Cartagena ', 'Cartagena de Indias', 'Cucuta', 'CÚCUTA', 'Cúcuta', 'Cúcuta ', 'IBAGUE', 'IBAGUÉ', 'Ibague', 'Ibagué', 'Ibagué ', 'MANIZALES', 'MDE', 'MEDELLIN', 'MEDELLÍN', 'MZL', 'Manizales', 'Manizales ', 'Manizales, Caldas', 'Medellin', 'Medellin, Ant.', 'Medellín', 'Medellín ', 'PEREIRA', 'Pereira', 'Pereira ', 'SANTA MARTA', 'SMR', 'Santa Marta', 'Santa Marta ', 'SantaMarta', 'Santiago de Cali', 'Sta. Marta', 'VILLAVICENCIO', 'Villavicencio', 'Villavicencio ', 'barranquilla', 'bogota', 'bogota d.c

Como esperaba de la auditoría: 20 valores para canal_compra y metodo_pago (deberían ser 4), 32 para categoria_producto (deberían ser 6) y 88 para ciudad_tienda (deberían ser 12). El problema es mayúsculas/espacios en las 4, y en categoria_producto y ciudad_tienda además hay tildes rotas por encoding y abreviaturas (BOG, MDE, CTG, "Bogota D.C.", "B/quilla"...).

## Paso 2 — Limpieza automática: espacios y mayúsculas

Aplico str.strip() y str.title() a las 4 columnas. A canal_compra y metodo_pago les alcanza, porque su único problema era de formato. A categoria_producto y ciudad_tienda no, porque ahí el lío es de encoding y abreviaturas, y eso ninguna función genérica lo adivina.

In [3]:
# se guarda ANTES de tocar nada, para poder confirmar mas adelante que la limpieza
# no introdujo nulos nuevos por accidente
nulos_antes_texto = df[columnas_texto].isna().sum()

for col in columnas_texto:
    df[col] = df[col].str.strip().str.title()

for col in columnas_texto:
    print(f"{col}: {df[col].nunique(dropna=True)} valores unicos tras strip()+title()")

canal_compra: 4 valores unicos tras strip()+title()
metodo_pago: 4 valores unicos tras strip()+title()
categoria_producto: 8 valores unicos tras strip()+title()
ciudad_tienda: 33 valores unicos tras strip()+title()


canal_compra y metodo_pago ya quedaron en 4 valores: con strip()+title() bastó. categoria_producto y ciudad_tienda siguen con más valores de la cuenta, así que pasan al Paso 3.

Un detalle curioso: title() convierte "PAYPAL" en "Paypal", no en "PayPal" (el nombre de marca lleva una P interna en mayúscula). Lo corrijo más abajo a mano — ni una limpieza "automática" se puede aplicar sin revisar el resultado.

## Paso 3 — Diccionario de mapeo: lo que la limpieza automática no resolvió

Para categoria_producto y ciudad_tienda armo una función que primero quita tildes y bytes de encoding roto con unicodedata, y luego resuelvo a mano, con un diccionario, las abreviaturas que quedan (BOG, MDE, CTG, BAQ, BGA, MZL, SMR; sufijos como "D.C." o ", Ant."; variantes como "B/quilla" o "Santiago de Cali").

In [4]:
def normaliza_texto(s):
    # Quita tildes/errores de encoding, espacios sobrantes y mayusculas para poder comparar variantes
    if pd.isna(s):
        return s
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().strip().lower()

# --- categoria_producto: una vez sin tildes rotas, cada categoria cae en una sola forma ---
mapa_categoria = {
    'belleza': 'Belleza', 'deportes': 'Deportes', 'electronica': 'Electronica',
    'hogar': 'Hogar', 'jugueteria': 'Jugueteria', 'moda': 'Moda',
}
df['categoria_producto'] = df['categoria_producto'].apply(normaliza_texto).map(mapa_categoria)

# --- ciudad_tienda: ademas de tildes rotas, hay abreviaturas de 3 letras, sufijos
# administrativos y variantes de escritura que no se resuelven solo quitando tildes ---
abreviaturas_ciudad = {
    'ctg': 'cartagena', 'baq': 'barranquilla', 'bga': 'bucaramanga', 'mde': 'medellin',
    'mzl': 'manizales', 'smr': 'santa marta', 'bog': 'bogota',
}

def normaliza_ciudad(s):
    if pd.isna(s):
        return s
    s2 = normaliza_texto(s).replace(',', ' ')
    for sufijo in [' d.c.', ' dc', ' ant.', ' caldas']:
        s2 = s2.replace(sufijo, '')
    s2 = s2.replace('b/quilla', 'barranquilla').replace('b/manga', 'bucaramanga')
    s2 = s2.replace('santiago de cali', 'cali').replace('cartagena de indias', 'cartagena')
    s2 = s2.replace('santamarta', 'santa marta').replace('sta. marta', 'santa marta').replace('sta marta', 'santa marta')
    s2 = ' '.join(s2.split())
    return abreviaturas_ciudad.get(s2, s2)

mapa_ciudad = {
    'bogota': 'Bogota', 'medellin': 'Medellin', 'cali': 'Cali', 'barranquilla': 'Barranquilla',
    'cartagena': 'Cartagena', 'bucaramanga': 'Bucaramanga', 'pereira': 'Pereira',
    'manizales': 'Manizales', 'ibague': 'Ibague', 'cucuta': 'Cucuta',
    'villavicencio': 'Villavicencio', 'santa marta': 'Santa Marta',
}
df['ciudad_tienda'] = df['ciudad_tienda'].apply(normaliza_ciudad).map(mapa_ciudad)

# correccion puntual del nombre de marca que title() dejo mal escrito
df['metodo_pago'] = df['metodo_pago'].replace('Paypal', 'PayPal')

for col in columnas_texto:
    print(f"{col}: {df[col].nunique(dropna=True)} valores unicos ->", sorted(df[col].dropna().unique()))

canal_compra: 4 valores unicos -> ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: 4 valores unicos -> ['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: 6 valores unicos -> ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: 12 valores unicos -> ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']


## Verificación — Pasos 2 y 3

Confirmo que las 4 columnas quedaron con una sola forma por categoría real, y sobre todo que no se coló ningún nulo nuevo: si map() se topa con algo que no anticipé en el diccionario, lo convierte en NaN en silencio, y esta verificación es la que lo atraparía.

In [5]:
for col in columnas_texto:
    assert df[col].nunique(dropna=True) in (4, 6, 12), f"{col} no quedo en su numero de categorias esperado"

nulos_despues_texto = df[columnas_texto].isna().sum()
diferencia = nulos_despues_texto - nulos_antes_texto
print("Nulos antes -> despues de la normalizacion de texto (debe ser 0 en todas):")
print(diferencia)
assert (diferencia == 0).all(), "La normalizacion introdujo nulos nuevos que no estaban en el dataset original"
print("\nOK: ninguna columna de texto gano nulos nuevos durante la limpieza.")

Nulos antes -> despues de la normalizacion de texto (debe ser 0 en todas):
canal_compra          0
metodo_pago           0
categoria_producto    0
ciudad_tienda         0
dtype: int64

OK: ninguna columna de texto gano nulos nuevos durante la limpieza.


## Paso 4 — fecha_pedido a un tipo de fecha real

fecha_pedido llega como texto en dos formatos (YYYY-MM-DD y DD/MM/YYYY). Antes de convertir reviso si hay nombres de mes en español mezclados (ej. "15 de enero de 2026"), porque pandas no los reconoce solo y tocaría reemplazarlos primero.

In [6]:
meses_es = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04', 'mayo': '05', 'junio': '06',
    'julio': '07', 'agosto': '08', 'septiembre': '09', 'setiembre': '09', 'octubre': '10',
    'noviembre': '11', 'diciembre': '12',
}

contiene_mes_texto = df['fecha_pedido'].astype(str).str.lower().str.contains('|'.join(meses_es.keys()), regex=True, na=False)
print(f"Filas con un nombre de mes en espanol dentro de fecha_pedido: {contiene_mes_texto.sum():,}")

Filas con un nombre de mes en espanol dentro de fecha_pedido: 0


No encontré ningún caso en este dataset: todas las fechas ya vienen en número. Aun así dejo el reemplazo de meses corriendo (no cambia nada aquí) en vez de asumir que nunca va a pasar — si llega una actualización del dataset con ese formato, ya queda cubierta.

In [7]:
fecha_pedido_texto = df['fecha_pedido'].astype(str).str.lower()
for mes_nombre, mes_numero in meses_es.items():
    fecha_pedido_texto = fecha_pedido_texto.str.replace(mes_nombre, mes_numero, regex=False)

nulos_antes_fecha = df['fecha_pedido'].isna().sum()

# se prueban los 2 formatos aceptados de forma explicita (nunca se deja que pandas "adivine")
fecha_iso = pd.to_datetime(fecha_pedido_texto, format='%Y-%m-%d', errors='coerce')
fecha_dmy = pd.to_datetime(fecha_pedido_texto, format='%d/%m/%Y', errors='coerce')
df['fecha_pedido'] = fecha_iso.fillna(fecha_dmy)

nulos_despues_fecha = df['fecha_pedido'].isna().sum()
print(f"fecha_pedido vacios ANTES de convertir (texto): {nulos_antes_fecha:,}")
print(f"fecha_pedido vacios (NaT) DESPUES de convertir a fecha real: {nulos_despues_fecha:,} ({nulos_despues_fecha/N*100:.2f}%)")
print(f"\ndtype final de fecha_pedido: {df['fecha_pedido'].dtype}")

fecha_pedido vacios ANTES de convertir (texto): 0
fecha_pedido vacios (NaT) DESPUES de convertir a fecha real: 22,687 (3.54%)

dtype final de fecha_pedido: datetime64[us]


El salto de 0 a 22,687 vacíos (3.54%) no lo produce esta conversión: pd.to_datetime() solo puede convertir fechas que existen de verdad, y lo que queda como NaT son fechas con formato inválido o de calendario inexistente (como "30/02/2025" o el mes 13) — el mismo número que ya había encontrado en la auditoría de Validez. Lo dejo como NaT en vez de inventar una fecha.

## Paso 5 — Duplicados: conteo antes de eliminar nada

Cuento dos tipos de duplicado por separado porque no son lo mismo:

1. Duplicados exactos: fila 100% idéntica.
2. Duplicados por correo_cliente: filas que comparten correo. Antes de decidir qué hacer, necesito entender por qué se repite.

In [8]:
dup_exactos = df.duplicated().sum()
print(f"Duplicados exactos (fila 100% identica): {dup_exactos:,} ({dup_exactos/N*100:.2f}%)")

placeholder = 'sin_registro@desconocido.com'
con_correo = df['correo_cliente'].notna()

dup_correo_total = df[con_correo].duplicated(subset='correo_cliente', keep=False).sum()
dup_correo_placeholder = (df['correo_cliente'] == placeholder).sum()
dup_correo_reales = dup_correo_total - dup_correo_placeholder

print(f"\nFilas que comparten correo_cliente con otra fila: {dup_correo_total:,} ({dup_correo_total/N*100:.2f}%)")
print(f"  - usan el valor constante '{placeholder}' (correo faltante imputado en la Mision 1): {dup_correo_placeholder:,}")
print(f"  - correos reales que se repiten: {dup_correo_reales:,}")

# la pregunta clave: cuando un correo real se repite, es la MISMA persona (varios pedidos
# legitimos) o corresponde a mas de un id_cliente distinto (lo que si seria una senal de error)?
correos_reales = df[con_correo & (df['correo_cliente'] != placeholder)]
clientes_por_correo = correos_reales.groupby('correo_cliente')['id_cliente'].nunique()
print(f"\nCorreos reales asociados a mas de un id_cliente distinto: {(clientes_por_correo > 1).sum():,} de {len(clientes_por_correo):,}")

Duplicados exactos (fila 100% identica): 20,000 (3.12%)

Filas que comparten correo_cliente con otra fila: 138,688 (21.67%)
  - usan el valor constante 'sin_registro@desconocido.com' (correo faltante imputado en la Mision 1): 105,312
  - correos reales que se repiten: 33,376



Correos reales asociados a mas de un id_cliente distinto: 0 de 518,000


De los correo_cliente repetidos, el 76% (105,312 de 138,688) es el valor constante "sin_registro@desconocido.com" que usé en la Misión 1 para los correos faltantes — no es la misma persona. Del resto, ningún correo real está asociado a más de un id_cliente: cuando un correo se repite es porque ese cliente compró más de una vez, no por un error de captura.

## Regla de conservación de duplicados

Duplicados exactos (20,000 filas, 3.12%): mismo id_pedido y todo lo demás igual — el mismo evento capturado dos veces. Como son copias idénticas, da igual cuál me quede: uso keep='first'.

Duplicados por correo_cliente: no los elimino solo por eso. Borrar por correo aquí destruiría historial de compra real de clientes recurrentes y colapsaría a una fila las 105,312 que comparten el marcador de correo faltante. Ningún correo real está ligado a más de un cliente, así que no hay señal de que compartir correo sea un error de identidad. Lo uso para verificar (Paso 6), no para eliminar.

In [9]:
print("Filas antes de eliminar duplicados exactos:", f"{len(df):,}")
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print("Filas despues de eliminar duplicados exactos:", f"{len(df):,}")

Filas antes de eliminar duplicados exactos: 640,000


Filas despues de eliminar duplicados exactos: 620,000


## Paso 6 — Verificación posterior

Confirmo que no quedan duplicados exactos, y que la repetición de correo_cliente que sigue ahí no esconde nada: busco filas con el mismo correo real, la misma fecha_pedido, el mismo producto y el mismo monto_compra a la vez — eso sí sería el mismo pedido repetido con otro id_pedido.

In [10]:
dup_exactos_post = df.duplicated().sum()
print(f"Duplicados exactos restantes: {dup_exactos_post}")

con_correo_post = df['correo_cliente'].notna()
dup_correo_post = df[con_correo_post].duplicated(subset='correo_cliente', keep=False).sum()
print(f"Filas que aun comparten correo_cliente: {dup_correo_post:,} (esperado: siguen siendo clientes recurrentes + el marcador de correo faltante)")

subset_pedido_equivalente = ['correo_cliente', 'fecha_pedido', 'producto', 'monto_compra']
duplicados_encubiertos = df[con_correo_post & (df['correo_cliente'] != placeholder)].duplicated(subset=subset_pedido_equivalente, keep=False).sum()
print(f"\nPosibles duplicados encubiertos (mismo correo real + misma fecha + mismo producto + mismo monto, con id_pedido distinto): {duplicados_encubiertos:,}")

assert dup_exactos_post == 0, "Quedaron duplicados exactos sin eliminar"
assert duplicados_encubiertos == 0, "Hay pedidos que parecen el mismo evento repetido bajo un id_pedido distinto"
print("\nOK: no quedan duplicados exactos, y no hay evidencia de duplicados encubiertos detras de la repeticion de correo_cliente.")

Duplicados exactos restantes: 0
Filas que aun comparten correo_cliente: 102,000 (esperado: siguen siendo clientes recurrentes + el marcador de correo faltante)



Posibles duplicados encubiertos (mismo correo real + misma fecha + mismo producto + mismo monto, con id_pedido distinto): 0

OK: no quedan duplicados exactos, y no hay evidencia de duplicados encubiertos detras de la repeticion de correo_cliente.


## Paso 7 — Exportar el dataset depurado

In [11]:
df.to_csv('S06_PD_Pallares_DatasetDepurado.csv', index=False)
print(f"Dataset depurado exportado: S06_PD_Pallares_DatasetDepurado.csv ({len(df):,} filas, {df.shape[1]} columnas)")

verificacion = pd.read_csv('S06_PD_Pallares_DatasetDepurado.csv')
print("\nDuplicados exactos en el archivo exportado:", verificacion.duplicated().sum())
for col in columnas_texto:
    print(f"{col}: {verificacion[col].nunique()} valores unicos ->", sorted(verificacion[col].dropna().unique()))

Dataset depurado exportado: S06_PD_Pallares_DatasetDepurado.csv (620,000 filas, 25 columnas)



Duplicados exactos en el archivo exportado: 0
canal_compra: 4 valores unicos -> ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: 4 valores unicos -> ['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: 6 valores unicos -> ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']


ciudad_tienda: 12 valores unicos -> ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']


## Conclusión

Sobre las 640,000 filas de entrada:

- canal_compra, metodo_pago, categoria_producto y ciudad_tienda quedaron cada una en su forma canónica (4, 4, 6 y 12 valores), sin perder un solo registro en el proceso.
- fecha_pedido ya es datetime real. 22,687 filas (3.54%) quedaron en NaT por fechas inválidas o inexistentes — el mismo número de la auditoría, no algo nuevo.
- Quité 20,000 duplicados exactos (3.12%). El dataset queda en 620,000 filas.
- correo_cliente repetido lo investigué y documenté, pero no lo usé para eliminar nada: es el marcador de correo faltante y clientes recurrentes, no errores. Verifiqué que no hay duplicados escondidos detrás.

S06_PD_Pallares_DatasetDepurado.csv queda listo para atípicos, escalado, codificación y transformaciones.